In [3]:
import pandas as pd


estimation_method_abbr = {
    "Linguistic Confidence": "Linguistic Conf.",
    "Semantic Uncertainty": "Semantic Unc.",
    "Self-Evaluation": "Self-Eval.",
    "Token Probability": "Token Prob."
}

def format_model(model):
    model_map = {
        "LLAMA-3.1-8B-INSTRUCT": "\\begin{tabular}{@{}c@{}}LLAMA-3.1-\\\\8B-Instruct\\end{tabular}",
        "META-LLAMA-3-8B-INSTRUCT": "\\begin{tabular}{@{}c@{}}LLAMA-3-\\\\8B-Instruct\\end{tabular}",
        "MISTRAL-7B-INSTRUCT-V0.3": "\\begin{tabular}{@{}c@{}}MISTRAL-7B-\\\\Instruct-V0.3\\end{tabular}",
        "QWEN2.5-7B-INSTRUCT": "\\begin{tabular}{@{}c@{}}QWEN-2.5-\\\\7B-Instruct\\end{tabular}",
        "QWEN3-8B": "QWEN3-8B",
        "GPT-OSS-20B": "GPT-OSS-20B" 
    }
    formatted_name = model_map.get(model)
    if formatted_name:
        return formatted_name
    return model

def generate_latex_table(csv_files):
    """
    Generate LaTeX longtable from CSV files with all metrics columns
    """
    all_data = []
    
    # Load and label datasets
    order = ["TriviaQA", "MMLU", "SQuAD 2.0"]
    
    for dataset_name in order:
        filename = csv_files.get(dataset_name)
        if filename:
            try:
                temp_df = pd.read_csv(filename)
                temp_df['Dataset'] = dataset_name
                all_data.append(temp_df)
            except FileNotFoundError:
                print(f"Warning: {filename} not found.")

    if not all_data:
        return "No data found."

    df = pd.concat(all_data, ignore_index=True)

    # Start LaTeX String using longtable
    latex = [
        "\\small",
        "\\setlength{\\tabcolsep}{3.6pt}",
        "\\renewcommand{\\arraystretch}{1.05}",
        "",
        "\\begin{longtable}{llcccccccc}",
        "\\caption{Accuracy, calibration and discrimination metrics of all evaluated models on TriviaQA, MMLU and SQuAD 2.0.}\\\\",
        "\\toprule",
        "Model & Estimation",
        "& \\multicolumn{2}{c}{Accuracy}",
        "& \\multicolumn{3}{c}{Calibration}",
        "& \\multicolumn{3}{c}{Discrimination} \\\\",
        "\\cmidrule(lr){3-4} \\cmidrule(lr){5-7} \\cmidrule(lr){8-10}",
        " & Method",
        " & Acc & Acc$_{\\text{Dist}}$",
        " & ECE & dECE & dECE$_{\\text{pt}}$",
        " & AUROC & dAUROC & dAUROC$_{\\text{pt}}$ \\\\",
        "\\midrule",
        "\\endfirsthead",
        "\\multicolumn{10}{l}{\\textit{(continued from previous page)}}\\\\",
        "\\toprule",
        "Model & Estimation",
        "& \\multicolumn{2}{c}{Accuracy}",
        "& \\multicolumn{3}{c}{Calibration}",
        "& \\multicolumn{3}{c}{Discrimination} \\\\",
        "\\cmidrule(lr){3-4} \\cmidrule(lr){5-7} \\cmidrule(lr){8-10}",
        " & Method",
        " & Acc & Acc$_{\\text{Dist}}$",
        " & ECE & dECE & dECE$_{\\text{pt}}$",
        " & AUROC & dAUROC & dAUROC$_{\\text{pt}}$ \\\\",
        "\\midrule",
        "\\endhead",
        # "\\midrule \\multicolumn{10}{r}{\\textit{(continued on next page)}}\\\\",
        "\\endfoot",
        "\\bottomrule",
        "\\endlastfoot",
        ""
    ]

    for dataset in order:
        if dataset not in df['Dataset'].unique():
            continue
        
        latex.append(f"\\multicolumn{{10}}{{c}}{{\\textbf{{{dataset}}}}} \\\\")
        latex.append("\\midrule")
        
        ds_df = df[df['Dataset'] == dataset]
        for model in ds_df['Model'].unique():
            model_df = ds_df[ds_df['Model'] == model]
            n_rows = len(model_df)
            
            for i, (_, row) in enumerate(model_df.iterrows()):
                # Multirow logic for the Model column
                model_fmt = format_model(model)
                model_cell = f"\\multirow{{{n_rows}}}{{*}}{{{model_fmt}}}" if i == 0 else ""
                
                # Format all metrics to 3 decimal places
                line = (
                    f"{model_cell} & {estimation_method_abbr.get(row['Estimation Method'], row['Estimation Method'])} & "
                    f"{row['Acc']:.3f} & {row['Acc_dist']:.3f} & "
                    f"{row['ECE']:.3f} & {row['dECE']:.3f} & {row['dECE_pt']:.3f} & "
                    f"{row['AUROC']:.3f} & {row['dAUROC']:.3f} & {row['dAUROC_pt']:.3f} \\\\"
                )
                latex.append(line)
            latex.append("\\midrule")

    latex.append("\\end{longtable}")
    return "\n".join(latex)

# Configuration
files = {
    "TriviaQA": "metrics_table_trivia_qa.csv",
    "MMLU": "metrics_table_mmlu.csv",
    "SQuAD 2.0": "metrics_table_squadv2.csv"
}

# Generate and print table
print(generate_latex_table(files))

\small
\setlength{\tabcolsep}{3.6pt}
\renewcommand{\arraystretch}{1.05}

\begin{longtable}{llcccccccc}
\caption{Accuracy, calibration and discrimination metrics of all evaluated models on TriviaQA, MMLU and SQuAD 2.0.}\\
\toprule
Model & Estimation
& \multicolumn{2}{c}{Accuracy}
& \multicolumn{3}{c}{Calibration}
& \multicolumn{3}{c}{Discrimination} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & Method
 & Acc & Acc$_{\text{Dist}}$
 & ECE & dECE & dECE$_{\text{pt}}$
 & AUROC & dAUROC & dAUROC$_{\text{pt}}$ \\
\midrule
\endfirsthead
\multicolumn{10}{l}{\textit{(continued from previous page)}}\\
\toprule
Model & Estimation
& \multicolumn{2}{c}{Accuracy}
& \multicolumn{3}{c}{Calibration}
& \multicolumn{3}{c}{Discrimination} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & Method
 & Acc & Acc$_{\text{Dist}}$
 & ECE & dECE & dECE$_{\text{pt}}$
 & AUROC & dAUROC & dAUROC$_{\text{pt}}$ \\
\midrule
\endhead
\endfoot
\bottomrule
\endlastfoot

\multicolumn{10}{c}{\t